# 加上新数据

In [2]:
import pandas as pd
df = pd.read_stata('事件日.dta')
df.to_excel('事件日.xlsx', index=False)

我重新计算了异常收益率

在此之前，我要提取数据，从CMSAR

## 标准化日期-股票代码

In [3]:
import pandas as pd

def process_stock_data(input_file, output_file):
    # 1. 读取数据 (请确保文件名和您的实际文件名一致)
    df = pd.read_excel(input_file)
    
    # 2. 股票代码前面填充0，使其足够六位
    # astype(str)将原本的数字转换为字符串，str.zfill(6)在左侧补0直到总长度为6
    df['股票代码'] = df['股票代码'].astype(str).str.zfill(6)
    
    # 3. 统一日期格式为 YYYY-MM-DD
    # pd.to_datetime 能够自动解析各种日期格式
    # dt.strftime('%Y-%m-%d') 将解析后的时间重新格式化为不含斜杠、不含时分秒的纯日期
    df['event date'] = pd.to_datetime(df['event date'], format='mixed').dt.strftime('%Y-%m-%d')
    
    # 4. 保存为新的 Excel (.xlsx) 文件
    # index=False 表示不保存最左侧的行号
    df.to_excel(output_file, index=False)
    
    print(f"处理完成！文件已保存至: {output_file}")

# 执行函数
if __name__ == "__main__":
    # 输入文件名保持为您的原CSV文件名
    input_file = '新数据.xlsx'   
    # 输出文件名改为 .xlsx 后缀
    output_file = '处理后_新数据.xlsx'
    
    process_stock_data(input_file, output_file)

处理完成！文件已保存至: 处理后_新数据.xlsx


现在，我提取股票代码，这些股票代码有着2014-2025的数据

In [4]:
import pandas as pd

def extract_unique_stock_codes(input_file, output_file):
    # 读取原始数据
    df = pd.read_excel(input_file)
    
    # 填充0至6位
    df['股票代码'] = df['股票代码'].astype(str).str.zfill(6)
    
    # 提取独特的股票代码 (去重)
    unique_stocks = df['股票代码'].unique()
    
    # 转换为 DataFrame 格式，方便导出
    unique_df = pd.DataFrame(unique_stocks, columns=['独特的股票代码'])
    
    # 保存为单独的 Excel 文件
    unique_df.to_excel(output_file, index=False)
    
    print(f"提取完成！共找到 {len(unique_stocks)} 个独特的股票代码，文件已保存至: {output_file}")

# 运行代码
if __name__ == "__main__":
    input_file = '处理后_新数据.xlsx'  # 您的原文件名
    output_file = '独特的股票代码.xlsx'      # 输出的新文件名
    
    extract_unique_stock_codes(input_file, output_file)

提取完成！共找到 707 个独特的股票代码，文件已保存至: 独特的股票代码.xlsx


In [5]:
import os
base_path = r"D:\mycodelife\workshop\fake_finance\graduate\abnormal_return"  
df_event = pd.read_excel(os.path.join(base_path, "事件日.xlsx"))
df_event.rename(columns={'股票代码': '证券代码', 'event date': '事件日期'}, inplace=True)
df_event['事件日期'] = pd.to_datetime(df_event['事件日期'])
df_event['证券代码'] = df_event['证券代码'].astype(str).str.extract(r'(\d{6})')[0]
def print_event_count(df, step_name):
    if '事件日期' in df.columns and '证券代码' in df.columns:
        # 去重计算独特的 (证券代码, 事件日期) 组合数
        unique_events = df[['证券代码', '事件日期']].drop_duplicates().shape[0]
        print(f" 🚦 [检查点 - {step_name}] 当前独特事件总数: {unique_events} 个")
    else:
        print(f" 🚦 [检查点 - {step_name}] 无法统计（缺少必要列）")
print_event_count(df_event, 1)

 🚦 [检查点 - 1] 当前独特事件总数: 887 个


In [10]:
import os
import pandas as pd

# ==========================================
# 1. 定义检查与打印函数
# ==========================================

def print_event_count(df, step_name):
    """统计并打印独特的 (证券代码, 事件日期) 组合总数"""
    if '事件日期' in df.columns and '证券代码' in df.columns:
        # 去重计算独特的 (证券代码, 事件日期) 组合数
        unique_events = df[['证券代码', '事件日期']].drop_duplicates().shape[0]
        print(f" 🚦 [检查点 - {step_name}] 当前独特事件总数: {unique_events} 个")
    else:
        print(f" 🚦 [检查点 - {step_name}] 无法统计（缺少必要列）")

def print_duplicate_rows(df):
    """打印所有 (证券代码, 事件日期) 重复的具体行数据 (包含原本的那一行)"""
    if '事件日期' in df.columns and '证券代码' in df.columns:
        duplicates = df[df.duplicated(subset=['证券代码', '事件日期'], keep=False)]
        if not duplicates.empty:
            duplicates = duplicates.sort_values(by=['证券代码', '事件日期'])
            print(f" ⚠️ 发现 {len(duplicates)} 行重复数据（含原始行），详情如下：")
            print(duplicates.to_string()) # 使用 to_string() 确保打印不被折叠
        else:
            print(" ✅ 数据很干净，没有发现重复的具体行。")
    else:
        print(" 🚦 无法检查重复（缺少必要列）")

def print_duplicate_counts(df):
    """打印重复的 (证券代码, 事件日期) 组合及其出现的具体频次"""
    if '事件日期' in df.columns and '证券代码' in df.columns:
        counts = df.groupby(['证券代码', '事件日期']).size().reset_index(name='出现次数')
        duplicate_combinations = counts[counts['出现次数'] > 1]
        
        if not duplicate_combinations.empty:
            print(f" ⚠️ 发现 {len(duplicate_combinations)} 个独特的组合存在重复，频次如下：")
            print(duplicate_combinations.sort_values(by='出现次数', ascending=False).to_string())
        else:
            print(" ✅ 没有发现重复的组合频次。")
    else:
        print(" 🚦 无法检查重复频次（缺少必要列）")


# ==========================================
# 2. 主执行逻辑
# ==========================================

if __name__ == "__main__":
    # 定义路径
    base_path = r"D:\mycodelife\workshop\fake_finance\graduate\abnormal_return"  
    file_path = os.path.join(base_path, "事件日.xlsx")

    # 检查文件是否存在
    if not os.path.exists(file_path):
        print(f" ❌ 错误：找不到文件，请确认路径是否正确: {file_path}")
    else:
        print(" ⏳ 正在读取数据...")
        # 读取 Excel
        df_event = pd.read_excel(file_path)

        # 数据清洗与格式化
        df_event.rename(columns={'股票代码': '证券代码', 'event date': '事件日期'}, inplace=True)
        df_event['事件日期'] = pd.to_datetime(df_event['事件日期'])
        df_event['证券代码'] = df_event['证券代码'].astype(str).str.extract(r'(\d{6})')[0]

        # -----------------------------------
        # 开始打印各项统计信息
        # -----------------------------------
        print("\n" + "="*40)
        print(" 📊 数据统计与诊断报告")
        print("="*40)

        # 1. 打印去重后的独特事件总数
        print("\n--- 步骤 1: 独特事件统计 ---")
        print_event_count(df_event, "数据清洗完成")

        # 2. 打印重复的具体行数据 (方便你肉眼排查是哪些行)
        print("\n--- 步骤 2: 查看重复的完整行数据 ---")
        print_duplicate_rows(df_event)

        # 3. 打印重复组合的出现次数 (宏观掌握重复的严重程度)
        print("\n--- 步骤 3: 查看重复组合及频次 ---")
        print_duplicate_counts(df_event)
        
        print("\n" + "="*40)

 ⏳ 正在读取数据...

 📊 数据统计与诊断报告

--- 步骤 1: 独特事件统计 ---
 🚦 [检查点 - 数据清洗完成] 当前独特事件总数: 887 个

--- 步骤 2: 查看重复的完整行数据 ---
 ⚠️ 发现 96 行重复数据（含原始行），详情如下：
       证券代码       事件日期
505  300104 2018-03-14
506  300104 2018-03-14
507  300104 2018-03-14
508  300104 2018-03-14
863  300522 2021-02-24
864  300522 2021-02-24
865  300522 2021-02-24
866  300522 2021-02-24
715  300773 2019-11-19
716  300773 2019-11-19
717  300773 2019-11-19
718  300773 2019-11-19
1       NaN 2015-01-04
2       NaN 2015-01-04
3       NaN 2015-01-07
4       NaN 2015-01-07
10      NaN 2015-01-26
11      NaN 2015-01-26
24      NaN 2015-03-12
25      NaN 2015-03-12
35      NaN 2015-04-20
36      NaN 2015-04-20
48      NaN 2015-06-15
49      NaN 2015-06-15
55      NaN 2015-07-07
57      NaN 2015-07-07
73      NaN 2015-08-21
74      NaN 2015-08-21
75      NaN 2015-08-22
76      NaN 2015-08-22
97      NaN 2015-11-08
98      NaN 2015-11-08
140     NaN 2016-04-12
142     NaN 2016-04-12
153     NaN 2016-05-05
155     NaN 2016-05-05
174     NaN 

## 计算出异常收益率

In [14]:
import os
import pandas as pd
import numpy as np

# ==========================================================
# 0. 路径与参数设置 (请修改为你的实际文件夹路径)
# ==========================================================
base_path = r"D:\mycodelife\workshop\fake_finance\graduate\abnormal_return" 
output_path = os.path.join(base_path, "输出")
os.makedirs(output_path, exist_ok=True)

# 窗口期设置 (例如：估计期 [-70, -11]，事件期 [-10, 10])
est_start, est_end = -70, -11
evt_start, evt_end = -10, 10

# ==========================================================
# 1. 读取并拼接个股收益率数据
# ==========================================================
print("1. 正在读取并拼接个股回报率数据...")

files = ["2014-2018.xlsx", "2019-2023.xlsx", "2024.xlsx"]
df_stocks = []

for file in files:
    file_path = os.path.join(base_path, file)
    df_temp = pd.read_excel(file_path)
    df_stocks.append(df_temp)

df_stock = pd.concat(df_stocks, ignore_index=True)

# 统一列名以方便后续处理
df_stock.rename(columns={
    '考虑现金红利再投资的日个股回报率': '个股回报率'
}, inplace=True)

# ==========================================================
# 2. 读取市场回报率并与个股合并
# ==========================================================
print("2. 正在读取市场回报率并按交易日期和市场类型匹配...")

df_market = pd.read_excel(os.path.join(base_path, "市场回报率.xlsx"))
df_market.rename(columns={
    '考虑现金红利再投资的日市场回报率(等权平均法)': '市场回报率'
}, inplace=True)

# 将个股数据与市场数据进行匹配 (双主键匹配)
df_trading = pd.merge(df_stock, df_market, on=['交易日期', '市场类型'], how='left')

# ==========================================================
# 3. 读取事件日并与交易数据关联
# ==========================================================
print("3. 正在读取事件日数据并关联行情...")

df_event = pd.read_excel(os.path.join(base_path, "事件日.xlsx"))
# 统一列名，使其与行情数据对齐
df_event.rename(columns={
    '股票代码': '证券代码',
    'event date': '事件日期'
}, inplace=True)

# 确保代码列的数据类型一致（统一转为字符串，防止前导0丢失导致匹配失败）
df_event['证券代码'] = df_event['证券代码'].astype(str).str.zfill(6)
df_trading['证券代码'] = df_trading['证券代码'].astype(str).str.zfill(6)

# 将事件表与行情表合并
# 注意：这里会把该股票的所有历史交易记录都拼接到该事件下
df_merged = pd.merge(df_event, df_trading, on='证券代码', how='inner')

# 确保时间列为 datetime 格式
df_merged['交易日期'] = pd.to_datetime(df_merged['交易日期'])
df_merged['事件日期'] = pd.to_datetime(df_merged['事件日期'])

# ==========================================================
# 4. 对齐交易日，生成相对时间轴 (dif)
# ==========================================================
print("4. 正在对齐时间轴并计算相对交易日(dif)...")

# 排序并生成连贯的交易日序号
df_merged = df_merged.sort_values(by=['证券代码', '事件日期', '交易日期'])
df_merged['date_num'] = df_merged.groupby(['证券代码', '事件日期']).cumcount() + 1

# 处理非交易日公告：计算交易日与事件日的差值
df_merged['间隔时间'] = (df_merged['交易日期'] - df_merged['事件日期']).dt.days

# 提取间隔时间 >= 0 的记录（事件发生当天或之后的交易日）
valid_dates = df_merged[df_merged['间隔时间'] >= 0].copy()
# 找到距离事件日最近的那个交易日
valid_dates['min_dif'] = valid_dates.groupby(['证券代码', '事件日期'])['间隔时间'].transform('min')

# 定位 t=0 的基准交易日序号 (td)
t0_mask = valid_dates['间隔时间'] == valid_dates['min_dif']
td_df = valid_dates[t0_mask].groupby(['证券代码', '事件日期'])['date_num'].first().reset_index()
td_df.rename(columns={'date_num': 'td'}, inplace=True)

# 将基准日序号合并回总表，计算 dif
df_merged = pd.merge(df_merged, td_df, on=['证券代码', '事件日期'], how='left')
df_merged['dif'] = df_merged['date_num'] - df_merged['td']

# ==========================================================
# 5. 截取所需窗口期数据并导出
# ==========================================================
print("5. 正在提取指定窗口期数据...")

# 只保留 估计窗口 和 事件窗口 内的数据
min_window = min(est_start, evt_start)
max_window = max(est_end, evt_end)

df_final = df_merged[(df_merged['dif'] >= min_window) & (df_merged['dif'] <= max_window)].copy()

# 生成唯一的事件ID（为了区分同一家公司在不同时间发生的多次事件）
df_final['事件ID'] = df_final.groupby(['证券代码', '事件日期']).ngroup() + 1

# 整理列的顺序，让输出表格更美观
cols_order = ['事件ID', '证券代码', '事件日期', '交易日期', 'dif', '市场类型', '个股回报率', '市场回报率']
df_final = df_final[cols_order]

# 导出最终文件
output_file = os.path.join(output_path, "用于计算异常收益率的基础数据.xlsx")
df_final.to_excel(output_file, index=False)

print(f"数据处理完成！最终文件已保存至：{output_file}")

1. 正在读取并拼接个股回报率数据...
2. 正在读取市场回报率并按交易日期和市场类型匹配...
3. 正在读取事件日数据并关联行情...
4. 正在对齐时间轴并计算相对交易日(dif)...
5. 正在提取指定窗口期数据...
数据处理完成！最终文件已保存至：D:\mycodelife\workshop\fake_finance\graduate\abnormal_return\输出\用于计算异常收益率的基础数据.xlsx


我开始计算异常收益率，这个代码是基于邓沛的stata修改的

In [15]:
import os
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from scipy import stats
import matplotlib.pyplot as plt

# ==========================================================
# 0. 全局设置与数据读取 (修改为当前目录)
# ==========================================================
current_dir = os.getcwd()
res_path = os.path.join(current_dir, "事件研究结果输出")
os.makedirs(res_path, exist_ok=True)

# 画图字体设置
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

print(f"1. 正在当前目录下读取基础数据...\n   工作路径: {current_dir}")
df_model = pd.read_excel("用于计算异常收益率的基础数据.xlsx")

# 统一主键名称并将收益率转化为百分比
if '事件ID' in df_model.columns:
    df_model.rename(columns={'事件ID': 'id'}, inplace=True)
df_model['个股回报率'] *= 100
df_model['市场回报率'] *= 100

# ==========================================================
# 1. 核心计算：执行回归、提取系数、计算各种 CAR 以及每日 AR
# ==========================================================
print("2. 正在执行稳健的 OLS 市场模型回归，并汇总个体 CAR、每日 AR 与系数...")

# 定义需要计算 CAR 的丰富区间
windows_to_calc = [
    (-10, 10), (-5, 5), (-4, 4), (-3, 3), (-2, 2), (-1, 1), 
    (0, 0), (-10, 0), (-5, 0), (-3, 0), (-1, 0), 
    (0, 1), (0, 3), (0, 5), (0, 10)
]

ar_raw_list = []       # 存储每日 AR 基础数据，用于画图和整体检验
summary_list = []      # 存储最终汇总的 Excel 结果表

for event_id, group in df_model.groupby('id'):
    # 划分估计期 [-70, -11] 和事件期 [-10, 10]
    est_window = group[(group['dif'] >= -70) & (group['dif'] <= -11)].dropna(subset=['个股回报率', '市场回报率'])
    evt_window = group[(group['dif'] >= -10) & (group['dif'] <= 10)].copy()
    
    # 样本量校验：估计期不足60天则剔除
    if len(est_window) < 60 or evt_window.empty:
        continue
        
    try:
        # 稳健拟合
        model = smf.ols('个股回报率 ~ 市场回报率', data=est_window)
        results = model.fit()  
        
        # 提取回归系数
        alpha = results.params['Intercept']
        beta = results.params['市场回报率']
        
        # 预测并计算每日 AR
        evt_window['predicted_return'] = results.predict(evt_window['市场回报率'])
        evt_window['AR'] = evt_window['个股回报率'] - evt_window['predicted_return']
        evt_window = evt_window.sort_values('dif')
        
        # 将每日 AR 数据存入列表
        ar_raw_list.append(evt_window[['id', '证券代码', '事件日期', '交易日期', 'dif', 'AR']])
        
        # 构建该事件的单行汇总数据
        summary_row = {
            '事件ID': event_id,  # 新增事件ID，方便双重差分合并主键
            '证券代码': group['证券代码'].iloc[0],
            '事件日期': group['事件日期'].iloc[0],
            'Alpha(截距项)': alpha,
            'Beta(市场系数)': beta
        }
        
        # 计算该事件在各个不同区间下的 CAR
        for start, end in windows_to_calc:
            mask = (evt_window['dif'] >= start) & (evt_window['dif'] <= end)
            car_value = evt_window.loc[mask, 'AR'].sum()
            col_name = f"CAR[{start},{end}]"
            summary_row[col_name] = car_value
            
        # 【新增需求】保存每一天的单独 AR
        for day in range(-10, 11):
            day_ar = evt_window.loc[evt_window['dif'] == day, 'AR']
            if not day_ar.empty:
                summary_row[f'AR_{day}'] = day_ar.values[0]
            else:
                summary_row[f'AR_{day}'] = np.nan # 应对停牌等数据缺失情况
                
        summary_list.append(summary_row)
        
    except np.linalg.LinAlgError:
        continue # 剔除矩阵奇异的异常样本

# 合并数据
df_ar = pd.concat(ar_raw_list, ignore_index=True)
df_summary = pd.DataFrame(summary_list)

# 导出 【核心需求：汇总 Excel】
summary_file = os.path.join(res_path, "1_个股事件CAR_AR与回归系数汇总表.xlsx")
df_summary.to_excel(summary_file, index=False)
print(f"   => 成功生成个体汇总表: {summary_file}")


# ==========================================================
# 2. 生成走势图 (基于全部样本的平均值 AAR 和 CAAR)
# ==========================================================
print("3. 正在生成整体走势图...")
df_ar['CAR'] = df_ar.groupby('id')['AR'].cumsum()
plot_data = df_ar.groupby('dif')[['AR', 'CAR']].mean().reset_index()

plt.figure(figsize=(10, 6))
plt.plot(plot_data['dif'], plot_data['AR'], color='orangered', linestyle='-', label='AAR (平均异常收益率)')
plt.plot(plot_data['dif'], plot_data['CAR'], color='blue', linestyle='--', label='CAAR (累计平均异常收益率)')

plt.xticks(np.arange(-10, 11, 2))
plt.xlabel('事件窗口期 (相对交易日)')
plt.ylabel('收益率 (%)')
plt.title('事件发生前后异常收益率走势分析')
plt.legend()
plt.grid(True, alpha=0.3)
plot_file = os.path.join(res_path, "2_总体AAR与CAAR走势图.png")
plt.savefig(plot_file, dpi=300, bbox_inches='tight')
plt.close()
print(f"   => 成功生成走势图: {plot_file}")

# ==========================================================
# 3. 总体 CAR 显著性检验
# ==========================================================
print("4. 正在执行总体区间显著性检验...")
def get_star(p_val):
    if p_val < 0.01: return '***'
    elif p_val < 0.05: return '**'
    elif p_val < 0.1: return '*'
    else: return ''

car_stats = []
for start, end in windows_to_calc:
    col_name = f"CAR[{start},{end}]"
    car_series = df_summary[col_name].dropna()
    
    if not car_series.empty:
        t_stat, p_val = stats.ttest_1samp(car_series, 0)
        car_stats.append({
            '分析区间': f"[{start}, {end}]", 
            'CAAR(平均数)': car_series.mean(), 
            't值': t_stat, 
            'p值': p_val, 
            '显著性': get_star(p_val)
        })

df_car_sig = pd.DataFrame(car_stats)
sig_file = os.path.join(res_path, "3_总体CAR显著性检验结果.xlsx")
df_car_sig.to_excel(sig_file, index=False)
print(f"   => 成功生成显著性检验表: {sig_file}")

print("\n🎉 完美收工！所有结果都已保存在当前目录下的 '事件研究结果输出' 文件夹中。")

1. 正在当前目录下读取基础数据...
   工作路径: d:\mycodelife\workshop\fake_finance\graduate
2. 正在执行稳健的 OLS 市场模型回归，并汇总个体 CAR、每日 AR 与系数...
   => 成功生成个体汇总表: d:\mycodelife\workshop\fake_finance\graduate\事件研究结果输出\1_个股事件CAR_AR与回归系数汇总表.xlsx
3. 正在生成整体走势图...
   => 成功生成走势图: d:\mycodelife\workshop\fake_finance\graduate\事件研究结果输出\2_总体AAR与CAAR走势图.png
4. 正在执行总体区间显著性检验...
   => 成功生成显著性检验表: d:\mycodelife\workshop\fake_finance\graduate\事件研究结果输出\3_总体CAR显著性检验结果.xlsx

🎉 完美收工！所有结果都已保存在当前目录下的 '事件研究结果输出' 文件夹中。


## 测度新增数据的情绪

In [2]:
import pandas as pd
from openai import OpenAI
import time
import openpyxl  # 用于实时保存

# 初始化客户端
client = OpenAI(
    api_key="sk-f60f2fc49b534b14b57dfa1893971bb1",
    base_url="https://api.deepseek.com"
)

# 定义情感分析函数
def get_sentiment(text):
    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[
                {"role": "system", "content": "你是一个财经新闻情感分析专家，请直接返回-1到1之间的数值，0表示中性，1表示正向，-1表示负向，只返回数字不要其他内容"},
                {"role": "user", "content": f"请分析以下文本的情感倾向（-1到1）：'{text}'"}
            ],
            temperature=0.1,
            max_tokens=5
        )
        
        # 提取数值结果
        result = response.choices[0].message.content.strip()
        return float(result)
    except Exception as e:
        print(f"分析失败：{str(e)}")
        return None

# 主处理流程
def process_excel(input_path, output_path):
    # 读取Excel
    df = pd.read_excel(input_path, engine='openpyxl')
    
    # 添加新列（如果不存在）
    if 'sentiment' not in df.columns:
        df['sentiment'] = None
    
    # 逐行处理
    for index, row in df.iterrows():
        if pd.isnull(row['sentiment']):  # 跳过已处理的行
            content = str(row['content'])
            
            # 获取情感分数
            score = get_sentiment(content)
            
            # 更新数据
            df.at[index, 'sentiment'] = score
            
            # 实时保存（使用openpyxl直接写入）
            with pd.ExcelWriter(output_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
                df.to_excel(writer, index=False)
            
            print(f"已处理第 {index+1} 行，分数：{score}")
            time.sleep(1)  # 防止API速率限制

# 运行
if __name__ == "__main__":
    input_file = "D:\mycodelife\workshop\\fake_finance\\graduate\\新增新闻的内容-用于情绪测度.xlsx"  # 输入文件路径
    output_file = "新增新闻文本情绪分数.xlsx"  # 输出文件路径
    
    # 初始化输出文件
    pd.DataFrame().to_excel(output_file, index=False)
    
    process_excel(input_file, output_file) 

已处理第 1 行，分数：-0.9
已处理第 2 行，分数：0.1
已处理第 3 行，分数：-0.8
已处理第 4 行，分数：-0.8
已处理第 5 行，分数：0.2
已处理第 6 行，分数：-0.9
已处理第 7 行，分数：0.1
已处理第 8 行，分数：0.0
已处理第 9 行，分数：0.2
已处理第 10 行，分数：-0.8
已处理第 11 行，分数：-0.2


# 合并数据

## 新新闻的控制变量合并

In [3]:
import pandas as pd
import numpy as np

# 1. 读取两个 Excel 文件
print("正在读取数据...")
df_news = pd.read_excel('新增新闻文本情绪分数.xlsx')
df_controls = pd.read_excel('新增数据的控制变量.xlsx')

# 2. 处理日期：提取年份
# 将 event date 转换为标准 datetime 格式，并提取年份存入新列 'Year'
df_news['event_date_dt'] = pd.to_datetime(df_news['event date'])
df_news['Year'] = df_news['event_date_dt'].dt.year

# 3. 合并数据表 (Left Join)
# 以新闻情绪表为主表，匹配控制变量表中的 '股票代码' 和 '会计年度'
print("正在按股票代码和年份合并数据...")
merged_df = pd.merge(
    df_news,
    df_controls,
    left_on=['股票代码', 'Year'],      # 第一个表的匹配键
    right_on=['股票代码', '会计年度'], # 第二个表的匹配键
    how='left'                       # 确保新闻情绪表的每一行都保留
)

# 4. 处理控制变量的列名与计算
print("正在计算控制变量并重命名为英文...")

# 如果下载的列名包含括号，先重命名为纯中文方便后续计算
if '总资产（企业规模）' in merged_df.columns:
    merged_df.rename(columns={'总资产（企业规模）': '总资产'}, inplace=True)

# A. 直接重命名虚拟变量和比率
rename_dict = {
    '产权性质': 'Soe',
    '两职合一': 'Dual',
    '独立董事比例': 'Indep',
    '审计师是否来自国际四大': 'Big4'
}
merged_df.rename(columns=rename_dict, inplace=True)

# B. 计算需要推导的财务指标
# 使用 np.log 计算自然对数，避免除以0或空值导致的报错
merged_df['Size'] = np.log(merged_df['总资产'].astype(float))
merged_df['CR'] = merged_df['流动资产'] / merged_df['流动负债']
merged_df['Lev'] = merged_df['总负债'] / merged_df['总资产']
merged_df['ND'] = merged_df['总负债'] / (merged_df['总资产'] - merged_df['总负债'])
merged_df['OPM'] = merged_df['营业利润'] / merged_df['营业收入']
merged_df['ARE'] = merged_df['净利润'] / merged_df['总资产']
merged_df['AT'] = merged_df['营业收入'] / merged_df['总资产']

# 5. 清理冗余列 (可选：如果您想保留原始财务数据，可以注释掉这一段)
# 删除用于合并的临时年份列、会计年度，以及计算完的原始中文财务列
columns_to_drop = [
    'event_date_dt', '会计年度', '总资产', '流动资产', '流动负债',
    '总负债', '营业利润', '营业收入', '净利润'
]
# 只删除数据框中实际存在的列，防止因列名不完全一致报错
cols_to_drop_actual = [col for col in columns_to_drop if col in merged_df.columns]
merged_df.drop(columns=cols_to_drop_actual, inplace=True)

# 6. 导出结果
output_filename = '合并后_情绪与控制变量.xlsx'
merged_df.to_excel(output_filename, index=False)
print(f"处理完成！文件已保存为: {output_filename}")

正在读取数据...
正在按股票代码和年份合并数据...
正在计算控制变量并重命名为英文...
处理完成！文件已保存为: 合并后_情绪与控制变量.xlsx


## 合并新旧数据

In [1]:
import pandas as pd

# 1. 读取旧数据和新数据
print("正在读取数据...")
df_old = pd.read_excel('旧数据.xlsx')
df_new = pd.read_excel('合并后_情绪与控制变量.xlsx')

# 2. 核心步骤：直接对比列名，找出新数据缺少的控制变量
# 用旧数据的列名集合 减去 新数据的列名集合
missing_vars = set(df_old.columns) - set(df_new.columns)

print("="*50)
print(f"新数据需要补充的控制变量（缺失列）共 {len(missing_vars)} 个：")
for var in missing_vars:
    print(f" - {var}")
print("="*50)

# 3. 将新数据合并（追加）到旧数据下方
# ignore_index=True 表示重新生成行号，避免新旧数据的原行号冲突
df_combined = pd.concat([df_old, df_new], ignore_index=True)

# 4. 导出合并后的文件，注意没有新异常收益率
output_filename = '新旧数据上下合并结果-初步合并.xlsx'
df_combined.to_excel(output_filename, index=False)
print(f"✅ 合并完成！合并后的文件已保存为: {output_filename}")

正在读取数据...
新数据需要补充的控制变量（缺失列）共 53 个：
 - variance
 - 交易日0
 - AR_6
 - AR_-8
 - CAR_-1_4
 - AR_9
 - AR_-1
 - AR_3
 - investor
 - relative_time
 - fuwei
 - AR_10
 - AR_-3
 - AR_-7
 - TR
 - Firm
 - AR_-10
 - sential
 - 市场类型
 - AR_-9
 - AR_7
 - 净利润
 - 董事会规模A
 - year
 - 独立董事人数
 - AR_-2
 - AR_-6
 - 分析师关注度
 - AR_-4
 - beta
 - CAR_-1_2
 - CAR_-1_5
 - AR_1
 - 总资产（企业规模）
 - is_authoritative
 - AR_-5
 - alpha
 - AR_5
 - CAR_-1_3
 - mean
 - 审计师是否来自国际四大
 - AR_0
 - Ind
 - l1_norm
 - CAR_-1_1
 - 年个股总市值
 - QR
 - AR_2
 - AR_8
 - final_score
 - AR_4
 - 总负债
 - 行业代码
✅ 合并完成！合并后的文件已保存为: 新旧数据上下合并结果-初步合并.xlsx


In [16]:
import pandas as pd

# ==========================================
# 1. 读取数据
# ==========================================
print("正在读取 Excel 文件...")
# 请将这里的文件名替换为你实际的文件路径
df1 = pd.read_excel('1_个股事件CAR_AR与回归系数汇总表.xlsx')
df2 = pd.read_excel('新旧数据上下合并结果-初步合并.xlsx')

# ==========================================
# 2. 核心防御：标准化主键格式（防止匹配失败）
# ==========================================
print("正在标准化主键格式...")
# 强制将股票代码转为 6 位字符串（补齐前导 0）
df1['股票代码'] = df1['股票代码'].astype(str).str.zfill(6)
df2['股票代码'] = df2['股票代码'].astype(str).str.zfill(6)

# 强制将 event date 统一转为时间格式，抹平字符串与时间戳的差异
df1['event date'] = pd.to_datetime(df1['event date'])
df2['event date'] = pd.to_datetime(df2['event date'])

# ==========================================
# 3. 按主键合并并保留所有列
# ==========================================
print("正在合并数据...")
# 使用 how='outer' 取并集，确保两张表里的任何记录都不丢失
# 如果有同名字段（除主键外），Pandas 会自动加上 _x 和 _y 后缀来区分
df_merged = pd.merge(df1, df2, on=['股票代码', 'event date'], how='outer')

# ==========================================
# 4. 根据主键去重
# ==========================================
print("正在去除重复记录...")
# subset 指定查重的列；keep='first' 表示如果遇到重复的，保留第一条
df_final = df_merged.drop_duplicates(subset=['股票代码', 'event date'], keep='first')

# ==========================================
# 5. 导出结果
# ==========================================
df_final.to_excel('除了一些罕见变量基本合并的数据.xlsx', index=False)
print(f"处理完成！合并前表1有 {len(df1)} 行，表2有 {len(df2)} 行。最终去重后保留了 {len(df_final)} 行。")

正在读取 Excel 文件...
正在标准化主键格式...
正在合并数据...
正在去除重复记录...
处理完成！合并前表1有 898 行，表2有 944 行。最终去重后保留了 927 行。
